# Loading a GeoParquet Ecosystem Map

This notebook demonstrates how to load a local GeoParquet file as an ecosystem map using the `iucn-get-data` package. No cloud credentials are required.

This complements `ecosystem_map_display.ipynb` (which uses Earth Engine backends) by showing the local-file Parquet workflow.

In [ ]:
from pathlib import Path

import iucn_get_data
from iucn_get_data import open_ecosystem_map, Typology

## Load a GeoParquet file

Use `open_ecosystem_map()` to load a local `.parquet` file. The Parquet backend is auto-detected from the file extension. Specify the column names that map to GET Level 3 (Ecosystem Functional Group) and Level 4/5/6 (ecosystem type) codes.

In [ ]:
# # Path to a sample GeoParquet file (bundled in the test data)
# parquet_path = str(Path('..') / 'tests' / 'test_data' / 'ruritania_ecosystems.parquet')

# ecosystems = open_ecosystem_map(
#     parquet_path,
#     get_level3_column='EFG1',
#     get_level456_column='Glob_Typol',
#     cmap={
#         'T1.1_NullIsland_forest_D01': [0, 128, 0],
#         'T6.5_NullIsland_alpine_D03': [255, 255, 255],
#         'M1.1_NullIsland_marine_shelf_D02': [0, 0, 255],
#     },
# )
# ecosystems


parquet_path = str(Path('..') / 'tests' / 'test_data' / 'colombian_ecoregions.parquet')
ecosystems = open_ecosystem_map(
    parquet_path,
    get_level3_column='EFG1',
    get_level456_column='COD',
    # cmap={
    #     'T1.1_NullIsland_forest_D01': [0, 128, 0],
    #     'T6.5_NullIsland_alpine_D03': [255, 255, 255],
    #     'M1.1_NullIsland_marine_shelf_D02': [0, 0, 255],
    # },
)
ecosystems

## Inspect the underlying GeoDataFrame

The `.data` property returns the full `geopandas.GeoDataFrame`, including geometry.

In [ ]:
ecosystems.data

## Display on an interactive map

Use the built-in `to_map()` method to render the ecosystem polygons on a [lonboard](https://developmentseed.org/lonboard/) deck.gl map directly in Jupyter.

In [ ]:
ecosystems.to_map(simplify_tolerance=0.001)

## Aggregated map views

Display maps aggregated to higher levels of the GET hierarchy. Geometries are dissolved and colored using defaults from `map_style.yaml`.

In [ ]:
# Functional Groups (GET Level 3) – dissolves by EFG code
ecosystems.to_functional_group_map(simplify_tolerance=0.001)

In [ ]:
# Biomes (GET Level 2) – dissolves by biome code derived from EFG
ecosystems.to_biome_map(simplify_tolerance=0.001)

In [ ]:
# Realms (GET Level 1) – dissolves by realm code derived from EFG
ecosystems.to_realm_map(simplify_tolerance=0.001)

## Extract the functional group DataFrame

`functional_group_dataframe()` drops geometry and duplicate rows, returning distinct ecosystem types indexed by GET Level 3 and Level 4/5/6 codes.

In [ ]:
fg_df = ecosystems.functional_group_dataframe()
fg_df

## Combine with the IUCN GET Typology

Pass the functional group DataFrame into `Typology` to see how the ecosystems fit into the full IUCN Global Ecosystem Typology hierarchy.

In [ ]:
typology = Typology(
    ecosystems=fg_df.reset_index(),
    ecosystems_functional_group_column='EFG1',
)
typology